# 🏥 Project 5 — Diabetes Hospital Readmission Prediction

## Recruiter-ready project overview

**Problem type:** Binary classification  
**UCI dataset:** Diabetes 130-US Hospitals for Years 1999–2008 (ID 296)

### Executive summary

This advanced project predicts whether a patient will be readmitted within 30 days after a hospital encounter. It demonstrates large, messy healthcare data preparation, missing-value handling, categorical encoding, class-imbalance strategies, leakage awareness, model comparison and model interpretation with feature importance and SHAP.

### What this project demonstrates

- Working with a large real-world dataset
- Missing-value and sparse-feature handling
- Categorical encoding
- Stratified train/test splitting
- Class imbalance
- Logistic Regression
- Random Forest
- XGBoost
- Precision, recall, F1 and ROC-AUC
- Threshold analysis
- Feature importance
- SHAP-based interpretation
- Data-leakage awareness
- Translating model results into operational implications

### The key data-science question

> **Can information available around a hospital encounter help identify patients at higher risk of readmission within 30 days?**

This is a binary classification problem because the outcome is represented as **readmitted within 30 days vs not within 30 days**.

### Critical modeling principle

Healthcare prediction is especially sensitive to **data leakage**. Variables that are only known after discharge or that directly encode the outcome can make a model appear much stronger than it would be in a real deployment. This project therefore explicitly discusses leakage-prone variables.

> **Educational/research project only — not a clinical diagnostic or treatment system.**

### Interview takeaway

A strong explanation is: **“This project pushed me beyond basic modeling. I had to think about messy data, class imbalance, leakage, threshold selection and interpretability. The main lesson was that a high metric is not enough; the features and information available at prediction time must make sense for the intended use.”**


# Project 5 — Diabetes Hospital Readmission Prediction

**Complexity:** Advanced  
**Problem type:** Binary Classification  
**Dataset:** UCI Diabetes 130-US Hospitals for Years 1999–2008 (UCI ID 296)

### What is happening?
This is the final and most advanced project in the sequence. We will predict whether a diabetic patient is readmitted to hospital within 30 days of discharge. The project introduces a large, messy dataset, missing-value placeholders, heavy categorical encoding, class imbalance, leakage checks, and model explainability.

## 1. Project objective

The objective is to identify patients at higher risk of 30-day readmission so healthcare teams can consider earlier follow-up and proactive care management.

### What is happening?
The target is converted into a binary problem: **1 = readmitted within 30 days** and **0 = not readmitted within 30 days**. The evaluation emphasizes recall, precision, F1, ROC-AUC, and the confusion matrix rather than accuracy alone.

## 2. Install and import libraries

### What is happening?
This section prepares Google Colab. We use UCI's `ucimlrepo` package for data access, scikit-learn for preprocessing/modeling, XGBoost for an additional non-linear model, and SHAP for explainability.

In [ ]:
!pip -q install ucimlrepo xgboost shap

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve
)

from xgboost import XGBClassifier

import shap

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

## 3. Load the UCI dataset

### What is happening?
The proposal identifies the Diabetes 130-US Hospitals dataset as UCI dataset **ID 296**. We retrieve its features and target, then inspect their shapes and available columns.

In [ ]:
diabetes = fetch_ucirepo(id=296)

X_raw = diabetes.data.features.copy()
y_raw = diabetes.data.targets.copy()

print("Feature shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

print("\nFeature columns:")
print(X_raw.columns.tolist())

print("\nTarget columns:")
print(y_raw.columns.tolist())

display(X_raw.head())
display(y_raw.head())

## 4. Combine predictors and target

### What is happening?
We create one working DataFrame. Because different versions of the UCI package can expose the target differently, the code checks the returned target structure before creating the working target column.

In [ ]:
df = X_raw.copy()

if y_raw.shape[1] == 1:
    df["readmitted_raw"] = y_raw.iloc[:, 0].values
else:
    possible_target = [c for c in y_raw.columns if "readmit" in c.lower()]
    if possible_target:
        df["readmitted_raw"] = y_raw[possible_target[0]].values
    else:
        raise ValueError("Could not identify the readmission target.")

print("Working shape:", df.shape)
display(df.head())

## 5. Inspect the raw data

### What is happening?
This dataset contains many categorical fields and uses placeholder values such as `?` for missing information. We inspect data types, missing markers, duplicates, and summary information before cleaning.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nRows containing '?' by column:")
question_counts = (df == "?").sum().sort_values(ascending=False)
display(question_counts.head(30))

print("\nDuplicate rows:", df.duplicated().sum())

## 6. Preserve the untouched raw dataset

### What is happening?
Keeping an untouched copy makes the workflow reproducible and lets us compare the original data with the cleaned version.

In [ ]:
df_raw = df.copy()

## 7. Inspect the target classes

### What is happening?
The original readmission target distinguishes `<30`, `>30`, and `NO`. We need to identify the exact labels before converting them to the binary research target.

In [ ]:
print(df["readmitted_raw"].value_counts(dropna=False))
print("\nTarget proportions:")
display(df["readmitted_raw"].value_counts(normalize=True, dropna=False))

## 8. Create the binary target

### What is happening?
Following the project's objective, `<30` becomes the positive class because it represents readmission within 30 days. Both `>30` and `NO` become the negative class.

In [ ]:
df["readmitted_30d"] = (
    df["readmitted_raw"].astype(str).str.strip() == "<30"
).astype(int)

print(df["readmitted_30d"].value_counts())
print("\nProportions:")
display(df["readmitted_30d"].value_counts(normalize=True))

## 9. Clean missing-value placeholders

### What is happening?
The proposal specifically calls out `?` placeholders. We convert them to real missing values so that scikit-learn imputers can handle them consistently.

In [ ]:
df = df.replace("?", np.nan)

print("Missing values after converting '?':")
missing_summary = df.isna().sum().sort_values(ascending=False)
display(missing_summary.head(30))

## 10. Remove extremely sparse and identifier fields

### What is happening?
The proposal recommends dropping sparse fields such as weight and warns about leakage-prone discharge information. Patient identifiers also should not be used as predictive features.

We remove `weight` because it is extremely sparse in the original dataset. We also remove encounter/patient identifiers and the original target columns.

In [ ]:
drop_cols = [
    "weight",
    "encounter_id",
    "patient_nbr",
    "readmitted_raw",
    "readmitted_30d"
]

drop_cols = [c for c in drop_cols if c in df.columns]

df_model = df.drop(columns=drop_cols).copy()

print("Dropped columns:")
print(drop_cols)

print("\nRemaining shape:", df_model.shape)

## 11. Check discharge-disposition leakage risk

### What is happening?
Some discharge dispositions can describe outcomes such as death or hospice. These fields may be inappropriate or leakage-prone for a 30-day readmission prediction task because they can encode information unavailable or incompatible with the intended prediction setting.

We inspect the available categories and exclude the field from the primary model if it contains such outcome-related information.

In [ ]:
if "discharge_disposition_id" in df_model.columns:
    print("Discharge disposition categories:")
    display(df_model["discharge_disposition_id"].value_counts(dropna=False).head(30))

if "discharge_disposition_id" in df_model.columns:
    df_model = df_model.drop(columns=["discharge_disposition_id"])
    print("Removed discharge_disposition_id from modeling features.")

## 12. Inspect the cleaned target and class balance

### What is happening?
After cleaning the predictors, we restore the binary target separately and visualize its imbalance. A minority positive class means accuracy can look good even when the model misses many readmissions.

In [ ]:
y = df["readmitted_30d"].copy()

print("Target distribution:")
display(y.value_counts().rename(index={0: "Not readmitted within 30 days", 1: "Readmitted within 30 days"}))

plt.figure(figsize=(7, 5))
plt.bar(["No 30-day readmission", "30-day readmission"], y.value_counts().sort_index().values)
plt.ylabel("Number of admissions")
plt.title("30-Day Readmission Class Distribution")
plt.xticks(rotation=10)
plt.show()

## 13. Demographic exploration

### What is happening?
The proposal asks for demographic breakdowns by age, gender, and race. These plots help establish the population represented in the dataset before modeling.

In [ ]:
for col in ["race", "gender", "age"]:
    if col in df.columns:
        counts = df[col].fillna("Missing").value_counts()

        plt.figure(figsize=(9, 5))
        plt.bar(counts.index.astype(str), counts.values)
        plt.xlabel(col)
        plt.ylabel("Admissions")
        plt.title(f"Admissions by {col}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

## 14. Readmission rate by demographic variables

### What is happening?
Counts alone do not tell us whether a group has a different readmission rate. Here we calculate the percentage of admissions with 30-day readmission for selected demographic variables.

In [ ]:
for col in ["race", "gender", "age"]:
    if col in df.columns:
        rates = df.groupby(col, dropna=False)["readmitted_30d"].mean().sort_values(ascending=False) * 100

        plt.figure(figsize=(9, 5))
        plt.bar(rates.index.astype(str), rates.values)
        plt.xlabel(col)
        plt.ylabel("30-day readmission rate (%)")
        plt.title(f"30-Day Readmission Rate by {col}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

        display(rates.to_frame("readmission_rate_percent"))

## 15. Treatment and medication patterns

### What is happening?
The proposal asks for treatment and medication-change analysis. We examine diabetes medications and medication-change status where those fields are available.

In [ ]:
medication_cols = [
    c for c in [
        "diabetesMed", "change", "insulin",
        "metformin", "repaglinide", "nateglinide",
        "glimepiride", "glipizide", "glyburide",
        "pioglitazone", "rosiglitazone", "acarbose",
        "miglitol", "tolazamide", "tolbutamide"
    ] if c in df.columns
]

for col in medication_cols[:8]:
    counts = df[col].fillna("Missing").value_counts()

    plt.figure(figsize=(8, 4))
    plt.bar(counts.index.astype(str), counts.values)
    plt.xlabel(col)
    plt.ylabel("Admissions")
    plt.title(f"{col} Distribution")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 16. Readmission rate by admission type

### What is happening?
Admission type can reflect differences in urgency and patient context. We calculate readmission rates across admission-type categories where the field is available.

In [ ]:
if "admission_type_id" in df.columns:
    rates = df.groupby("admission_type_id")["readmitted_30d"].agg(["mean", "count"])
    rates["readmission_rate_percent"] = rates["mean"] * 100
    display(rates.sort_values("readmission_rate_percent", ascending=False))

    plt.figure(figsize=(8, 5))
    plt.bar(rates.index.astype(str), rates["readmission_rate_percent"])
    plt.xlabel("Admission type ID")
    plt.ylabel("30-day readmission rate (%)")
    plt.title("30-Day Readmission Rate by Admission Type")
    plt.show()

## 17. Readmission rate by length of stay

### What is happening?
Longer stays may indicate more complex admissions. We compare 30-day readmission rates across length-of-stay values.

In [ ]:
if "time_in_hospital" in df.columns:
    los = df.groupby("time_in_hospital")["readmitted_30d"].mean() * 100

    plt.figure(figsize=(10, 5))
    plt.plot(los.index, los.values, marker="o")
    plt.xlabel("Days in hospital")
    plt.ylabel("30-day readmission rate (%)")
    plt.title("Readmission Rate by Length of Stay")
    plt.grid(True, alpha=0.25)
    plt.show()

## 18. Readmission rate by prior visits

### What is happening?
Prior inpatient, emergency, and outpatient visits can represent previous healthcare utilization. We compare readmission rates across these variables.

In [ ]:
visit_cols = [
    c for c in [
        "number_inpatient",
        "number_emergency",
        "number_outpatient"
    ] if c in df.columns
]

for col in visit_cols:
    rates = df.groupby(col)["readmitted_30d"].agg(["mean", "count"])
    rates["rate_percent"] = rates["mean"] * 100

    display(rates.head(15))

    plt.figure(figsize=(10, 5))
    plt.plot(rates.index, rates["rate_percent"], marker="o")
    plt.xlabel(col)
    plt.ylabel("30-day readmission rate (%)")
    plt.title(f"Readmission Rate by {col}")
    plt.grid(True, alpha=0.25)
    plt.show()

## 19. Create a modeling DataFrame

### What is happening?
We now combine the cleaned predictors with the binary target. The predictors remain mostly categorical/numeric mixed data, so the next step is automated preprocessing.

In [ ]:
model_df = df_model.copy()
model_df["readmitted_30d"] = y.values

X = model_df.drop(columns=["readmitted_30d"])
y = model_df["readmitted_30d"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 20. Identify numeric and categorical variables

### What is happening?
Numeric features will be median-imputed and standardized. Categorical features will be most-frequent imputed and one-hot encoded.

In [ ]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

## 21. Stratified train/test split

### What is happening?
The proposal specifically recommends a stratified split. This preserves approximately the same 30-day readmission ratio in both training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

## 22. Build preprocessing pipelines

### What is happening?
One-hot encoding can create thousands of sparse features in this dataset. The pipeline handles this automatically and ensures that preprocessing is learned only from the training data.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 23. Define evaluation metrics

### What is happening?
For this problem, false negatives are important because they represent patients who are readmitted within 30 days but were not identified as high risk. We therefore report precision, recall, F1, ROC-AUC, and accuracy.

In [ ]:
def evaluate_classifier(model, X_eval, y_eval, threshold=0.5):
    probabilities = model.predict_proba(X_eval)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "F1": f1_score(y_eval, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities)
    }

## 24. Model 1 — Logistic Regression baseline

### What is happening?
Logistic Regression is the interpretable baseline. We use class weighting so the minority 30-day readmission class receives more attention during training.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_results = evaluate_classifier(
    logistic_model, X_test, y_test
)

display(pd.DataFrame([logistic_results], index=["Logistic Regression"]))

## 25. Model 2 — Random Forest

### What is happening?
Random Forest captures non-linear relationships and interactions. Class weighting is again used to reduce the tendency to favor the majority class.

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample",
        min_samples_leaf=2
    ))
])

rf_model.fit(X_train, y_train)

rf_results = evaluate_classifier(
    rf_model, X_test, y_test
)

display(pd.DataFrame([rf_results], index=["Random Forest"]))

## 26. Model 3 — XGBoost

### What is happening?
XGBoost provides another powerful non-linear classifier. `scale_pos_weight` is calculated from the training class ratio so the minority readmission class receives additional weight.

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / max(positive_count, 1)

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)

xgb_results = evaluate_classifier(
    xgb_model, X_test, y_test
)

display(pd.DataFrame([xgb_results], index=["XGBoost"]))

## 27. Compare model performance

### What is happening?
The three models are compared using the evaluation metrics required by the proposal. There is no single universally best metric: recall is especially important when missing a true readmission is costly.

In [ ]:
results = pd.DataFrame({
    "Logistic Regression": logistic_results,
    "Random Forest": rf_results,
    "XGBoost": xgb_results
}).T

display(results.sort_values("ROC-AUC", ascending=False))

## 28. Cross-validation on the training data

### What is happening?
A single holdout can be sensitive to the particular split. Stratified 5-fold cross-validation provides a more stable estimate while keeping the class ratio approximately consistent across folds.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

cv_rows = []

for name, model in cv_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_rows.append({
        "Model": name,
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV ROC-AUC", ascending=False)
display(cv_results)

## 29. Confusion matrix for the best ROC-AUC model

### What is happening?
We select the model with the highest holdout ROC-AUC and inspect its confusion matrix. This makes false negatives and false positives visible rather than hiding them inside a single score.

In [ ]:
best_name = results["ROC-AUC"].idxmax()
best_model = cv_models[best_name]

best_pred = best_model.predict(X_test)

print("Best model by holdout ROC-AUC:", best_name)
print("\nClassification report:")
print(classification_report(
    y_test,
    best_pred,
    target_names=["No 30-day readmission", "30-day readmission"],
    zero_division=0
))

cm = confusion_matrix(y_test, best_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No 30-day", "30-day"]
)

disp.plot()
plt.title(f"Confusion Matrix — {best_name}")
plt.show()

## 30. ROC curves

### What is happening?
ROC curves show the trade-off between sensitivity and false-positive rate across different classification thresholds. ROC-AUC summarizes this ranking performance.

In [ ]:
models_for_roc = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

plt.figure(figsize=(9, 6))

for name, model in models_for_roc.items():
    probabilities = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — 30-Day Readmission")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

## 31. Examine an alternative decision threshold

### What is happening?
A 0.50 probability threshold is not automatically the best choice when recall is important. We inspect several thresholds to show how recall and precision change.

In [ ]:
threshold_rows = []

probabilities = best_model.predict_proba(X_test)[:, 1]

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    preds = (probabilities >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results)

## 32. Random Forest feature importance

### What is happening?
Feature importance provides a first interpretation of which transformed variables contribute most strongly to the Random Forest's predictions.

In [ ]:
rf_preprocessor = rf_model.named_steps["preprocessor"]
rf_estimator = rf_model.named_steps["model"]

rf_feature_names = rf_preprocessor.get_feature_names_out()
rf_importances = rf_estimator.feature_importances_

rf_importance_df = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_importances
}).sort_values("importance", ascending=False)

display(rf_importance_df.head(25))

## 33. Plot the strongest Random Forest predictors

### What is happening?
This visualization turns the feature-importance table into a report-friendly chart. Remember that one original categorical variable can produce many one-hot encoded features.

In [ ]:
top_rf = rf_importance_df.head(20).sort_values("importance")

plt.figure(figsize=(11, 8))
plt.barh(top_rf["feature"], top_rf["importance"])
plt.xlabel("Importance")
plt.ylabel("Encoded feature")
plt.title("Top Random Forest Predictors of 30-Day Readmission")
plt.tight_layout()
plt.show()

## 34. SHAP explainability for XGBoost

### What is happening?
The proposal requests feature importance / SHAP. We use SHAP to explain the XGBoost model. Because the dataset expands substantially after one-hot encoding, we sample the test data for a practical Colab explanation run.

In [ ]:
xgb_preprocessor = xgb_model.named_steps["preprocessor"]
xgb_estimator = xgb_model.named_steps["model"]

X_test_transformed = xgb_preprocessor.transform(X_test)

sample_size = min(1500, X_test_transformed.shape[0])
rng = np.random.default_rng(42)
sample_idx = rng.choice(
    X_test_transformed.shape[0],
    size=sample_size,
    replace=False
)

X_shap = X_test_transformed[sample_idx]

feature_names_xgb = xgb_preprocessor.get_feature_names_out()

print("Transformed feature count:", len(feature_names_xgb))
print("SHAP sample size:", X_shap.shape[0])

## 35. Generate the SHAP summary

### What is happening?
The SHAP summary ranks features by their average contribution to the model's predictions. Positive SHAP values push predictions toward 30-day readmission, while negative values push them away.

In [ ]:
explainer = shap.TreeExplainer(xgb_estimator)
shap_values = explainer.shap_values(X_shap)

if hasattr(shap_values, "values"):
    shap_values_plot = shap_values.values
else:
    shap_values_plot = shap_values

shap.summary_plot(
    shap_values_plot,
    X_shap,
    feature_names=feature_names_xgb,
    max_display=20,
    show=False
)

plt.title("SHAP Summary — XGBoost")
plt.tight_layout()
plt.show()

## 36. Aggregate SHAP importance by encoded feature

### What is happening?
We calculate mean absolute SHAP values and rank the transformed predictors. This creates a numerical summary that can be saved alongside the other model outputs.

In [ ]:
mean_abs_shap = np.abs(shap_values_plot).mean(axis=0)

shap_importance_df = pd.DataFrame({
    "feature": feature_names_xgb,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance_df.head(25))

## 37. Save project outputs

### What is happening?
The proposal requires a reproducible notebook and cleaned outputs. We save the cleaned modeling data, model comparison tables, threshold analysis, Random Forest importance, and SHAP importance.

In [ ]:
output_dir = "diabetes_readmission_outputs"
os.makedirs(output_dir, exist_ok=True)

cleaned_output = model_df.copy()
cleaned_output.to_csv(
    f"{output_dir}/diabetes_readmission_cleaned_model_data.csv",
    index=False
)

results.to_csv(
    f"{output_dir}/model_comparison.csv"
)

cv_results.to_csv(
    f"{output_dir}/cross_validation_results.csv",
    index=False
)

threshold_results.to_csv(
    f"{output_dir}/threshold_analysis.csv",
    index=False
)

rf_importance_df.to_csv(
    f"{output_dir}/random_forest_feature_importance.csv",
    index=False
)

shap_importance_df.to_csv(
    f"{output_dir}/xgboost_shap_importance.csv",
    index=False
)

print("Saved:")
print("\n".join(sorted(os.listdir(output_dir))))

## 38. Build a concise results summary

### What is happening?
This creates a final table suitable for a project report or presentation. The values come directly from the models you ran rather than from predetermined numbers.

In [ ]:
summary = results.copy()

summary["Model"] = summary.index
summary = summary[[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]].sort_values("ROC-AUC", ascending=False)

display(summary)

## 39. Clinical/business storytelling

### What is happening?
The final step is to translate model performance and important predictors into a practical narrative. Do not claim that a feature causes readmission simply because the model finds it predictive.

Your write-up should answer:
- Which model performed best?
- What was its recall?
- How many 30-day readmissions were missed?
- Which variables were most influential?
- What types of patients might benefit from closer follow-up?
- How could earlier identification potentially support care-management planning?

## 40. Cost-saving and care-management interpretation

### What is happening?
The proposal frames the project around proactive care and potentially reducing avoidable readmissions. The correct conclusion is evidence-based: the model can be presented as a **risk-stratification aid**, not as a replacement for clinical judgment.

Possible recommendations, if supported by your results:
- prioritize follow-up for patients flagged as higher risk;
- coordinate post-discharge medication and appointment support;
- review frequent prior utilization as part of discharge planning;
- use risk scores to help allocate limited care-management resources;
- monitor false negatives because missed high-risk patients are particularly important.

## 41. Limitations

### What is happening?
A strong data-science project reports limitations rather than presenting the model as perfect.

Important limitations to discuss:
- The dataset represents historical admissions from US hospitals during 1999–2008.
- Some variables have substantial missingness and required removal or imputation.
- Categorical encoding creates a very high-dimensional feature space.
- Class imbalance makes accuracy an incomplete evaluation measure.
- Discharge-related variables can create leakage or unrealistic prediction settings and were therefore handled cautiously.
- Feature importance shows predictive association, not causation.
- A production clinical system would require external validation, calibration, governance, privacy controls, and clinical oversight.

## 42. Final conclusion template

### What is happening?
Replace the bracketed values with the actual results from your notebook after execution.

**Conclusion**

This project developed a machine-learning pipeline for predicting 30-day hospital readmission among diabetic patients using the UCI Diabetes 130-US Hospitals dataset. The workflow addressed missing-value placeholders, sparse variables, categorical encoding, class imbalance, and potential information leakage.

Three models — Logistic Regression, Random Forest, and XGBoost — were trained and compared using precision, recall, F1-score, ROC-AUC, and a confusion matrix. The best-performing model was **[model]**, with a ROC-AUC of **[value]** and recall of **[value]** on the test set.

The interpretability analysis identified **[top predictors]** as important predictive signals. These findings suggest that machine-learning risk stratification could help care-management teams identify patients who may warrant additional post-discharge attention. However, the model should be treated as decision support rather than a clinical diagnosis, and further external validation would be required before real-world deployment.

# Project 5 completion checklist

### What is happening?
Before submitting the project, verify every major requirement from the proposal.

- [ ] UCI Diabetes dataset ID 296 loaded.
- [ ] Raw dataset preserved.
- [ ] `?` missing-value placeholders handled.
- [ ] Sparse `weight` field addressed.
- [ ] Identifier fields removed.
- [ ] Binary 30-day readmission target created.
- [ ] Discharge-disposition leakage risk reviewed.
- [ ] Demographic EDA completed.
- [ ] Treatment/medication EDA completed.
- [ ] Admission-type readmission analysis completed.
- [ ] Length-of-stay readmission analysis completed.
- [ ] Prior-visit readmission analysis completed.
- [ ] Stratified train/test split used.
- [ ] Logistic Regression baseline trained.
- [ ] Random Forest trained.
- [ ] XGBoost trained.
- [ ] Class imbalance addressed with weighting.
- [ ] Precision, Recall, F1 and ROC-AUC reported.
- [ ] Confusion matrix created.
- [ ] ROC curves created.
- [ ] Random Forest feature importance completed.
- [ ] SHAP summary completed.
- [ ] Outputs saved.
- [ ] Cost-saving/care-management story written.
- [ ] Limitations documented.

## 🎯 Portfolio conclusion — Project 5

This project demonstrates advanced applied machine learning with messy, imbalanced and leakage-sensitive data.

### Knowledge checkpoints

- **Class imbalance:** one target class is much more common than another.
- **Class weighting:** give more importance to minority-class mistakes during training.
- **Data leakage:** information from outside the legitimate prediction process makes evaluation unrealistically optimistic.
- **Threshold:** the probability cutoff used to convert a score into a class prediction.
- **SHAP:** explains how individual features contribute to a model's predictions.

### What to remember

In high-stakes domains, model development is not only about maximizing a score. **Data availability, leakage, threshold choice, interpretability and limitations are part of the machine-learning solution.**
